# Solar Cycle Data Exploration

This notebook explores the solar sunspot data and visualizes key patterns.

## Contents
1. Load and inspect data
2. Visualize solar cycles
3. Feature distributions
4. Time series analysis
5. Correlation analysis

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Libraries imported")

## 1. Load Data

In [ ]:
# Load raw daily sunspot data
raw_data = pd.read_csv('../data/raw/sidc_sunspot_daily.csv', parse_dates=['date'])

print(f"Raw Data Shape: {raw_data.shape}")
print(f"Date Range: {raw_data['date'].min()} to {raw_data['date'].max()}")
print(f"\nColumns: {list(raw_data.columns)}")

raw_data.head()

In [ ]:
# Load processed features
features = pd.read_csv('../data/features/daily_features.csv', parse_dates=['date'])

print(f"Features Shape: {features.shape}")
print(f"Number of Features: {features.shape[1]}")
print(f"Date Range: {features['date'].min()} to {features['date'].max()}")

features.head()

## 2. Visualize Solar Cycles

In [ ]:
# Plot full sunspot history
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(raw_data['date'], raw_data['ssn'], alpha=0.5, linewidth=0.5, label='Daily SSN')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Sunspot Number', fontsize=12)
ax.set_title('Sunspot Number History (1749-Present)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Maximum SSN: {raw_data['ssn'].max():.1f}")
print(f"Mean SSN: {raw_data['ssn'].mean():.1f}")
print(f"Median SSN: {raw_data['ssn'].median():.1f}")

In [ ]:
# Plot recent cycles (1980-present)
recent_data = raw_data[raw_data['year'] >= 1980]

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(recent_data['date'], recent_data['ssn'], linewidth=1, label='Daily SSN')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Sunspot Number', fontsize=12)
ax.set_title('Recent Solar Cycles (1980-Present)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Mark cycle maxima
cycle_maxima = [
    ('1989-11-01', 'Cycle 22'),
    ('2000-04-01', 'Cycle 23'),
    ('2014-04-01', 'Cycle 24')
]

for date, label in cycle_maxima:
    ax.axvline(pd.Timestamp(date), color='red', linestyle='--', alpha=0.5)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1] * 0.95, label, 
            rotation=90, verticalalignment='top', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Feature Distributions

In [ ]:
# Distribution of sunspot numbers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(raw_data['ssn'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sunspot Number', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('SSN Distribution', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Box plot by cycle phase
if 'cycle_phase' in features.columns:
    features.boxplot(column='ssn', by='cycle_phase', ax=axes[1])
    axes[1].set_xlabel('Cycle Phase', fontsize=12)
    axes[1].set_ylabel('Sunspot Number', fontsize=12)
    axes[1].set_title('SSN by Cycle Phase', fontsize=13, fontweight='bold')
    plt.suptitle('')  # Remove automatic title

plt.tight_layout()
plt.show()

## 4. Time Series Analysis

In [ ]:
# Autocorrelation
from pandas.plotting import autocorrelation_plot

fig, ax = plt.subplots(figsize=(14, 5))
autocorrelation_plot(raw_data['ssn'].dropna(), ax=ax)
ax.set_title('Sunspot Number Autocorrelation', fontsize=14, fontweight='bold')
ax.set_xlabel('Lag (days)', fontsize=12)
ax.set_ylabel('Autocorrelation', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Strong autocorrelation indicates predictability!")

In [ ]:
# Seasonality and trend decomposition
from statsmodels.tsa.seasonal import seasonal_decompose

# Resample to monthly for cleaner decomposition
monthly_data = raw_data.set_index('date').resample('MS')['ssn'].mean()

# Decompose (last 20 years)
recent_monthly = monthly_data[monthly_data.index >= '2003-01-01']
decomposition = seasonal_decompose(recent_monthly, model='additive', period=132)  # 11-year cycle

fig, axes = plt.subplots(4, 1, figsize=(16, 10))

decomposition.observed.plot(ax=axes[0], title='Observed')
decomposition.trend.plot(ax=axes[1], title='Trend')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal (11-year)')
decomposition.resid.plot(ax=axes[3], title='Residual')

for ax in axes:
    ax.set_ylabel('SSN')
    ax.grid(True, alpha=0.3)

plt.suptitle('Time Series Decomposition', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## 5. Feature Correlation Analysis

In [ ]:
# Select numerical columns for correlation
numerical_cols = features.select_dtypes(include=[np.number]).columns.tolist()

# Remove high cardinality columns
exclude_cols = ['year', 'month', 'day_of_year', 'days_since_1749', 'days_in_cycle']
numerical_cols = [col for col in numerical_cols if col not in exclude_cols]

# Compute correlation matrix
correlation_matrix = features[numerical_cols].corr()

# Get top correlated features with SSN
ssn_correlations = correlation_matrix['ssn'].abs().sort_values(ascending=False)

print("Top 15 Features Correlated with SSN:")
print(ssn_correlations.head(15))

In [ ]:
# Correlation heatmap (top features)
top_features = ssn_correlations.head(10).index.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(features[top_features].corr(), annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Feature Correlation Heatmap (Top 10)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
# Summary statistics
print("=" * 60)
print("SOLAR CYCLE DATA SUMMARY")
print("=" * 60)
print(f"\nData Coverage: {raw_data['date'].min()} to {raw_data['date'].max()}")
print(f"Total Days: {len(raw_data):,}")
print(f"Years of Data: {(raw_data['date'].max() - raw_data['date'].min()).days / 365.25:.1f}")

print(f"\nSunspot Number Statistics:")
print(f"  Mean: {raw_data['ssn'].mean():.1f}")
print(f"  Median: {raw_data['ssn'].median():.1f}")
print(f"  Std Dev: {raw_data['ssn'].std():.1f}")
print(f"  Min: {raw_data['ssn'].min():.1f}")
print(f"  Max: {raw_data['ssn'].max():.1f}")

print(f"\nFeature Engineering:")
print(f"  Total Features: {features.shape[1]}")
print(f"  Samples: {features.shape[0]:,}")
print(f"  Missing Values: {features.isnull().sum().sum()}")

print("\n" + "=" * 60)
print("Ready for model training!")
print("=" * 60)

## Next Steps

1. **Model Development** - See `02_model_development.ipynb`
2. **Hyperparameter Tuning** - See `03_hyperparameter_tuning.ipynb`
3. **Model Evaluation** - See `04_model_evaluation.ipynb`

To train models:
```bash
python ../src/training/train_short_term.py
```